# Train Microbiome PFN on a T4 GPU

End-to-end pretraining of the Microbiome PFN on a single **T4** (or similar 16 GB) GPU —
e.g. Google Colab's free T4 runtime.

**First:** in Colab, set **Runtime → Change runtime type → Hardware accelerator → T4 GPU**, then run the cells top to bottom.

Training uses **mixed precision (fp16 AMP)** automatically on CUDA (≈50% less VRAM and a meaningful speedup vs fp32). The settings below are a short *demo* (~2k steps); a serious model needs far longer — the README notes ~1M+ steps — and a bigger trunk (`d=512`, `n_layers=8`).

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — set Runtime > Change runtime type > T4 GPU, then restart.")

## 2. Install Microbiome PFN

Installs straight from GitHub. On Colab `torch` is already present (the CUDA build), so pip reuses it instead of downloading a fresh wheel.

In [ ]:
%pip install -q "git+https://github.com/metagenAu/microbiomepfn.git"
import microbiomepfn
from microbiomepfn.train import train
print("microbiomepfn", microbiomepfn.__version__)

## 3. Pretrain

Single-draw-per-step training on the validated prior. `n_taxa_cap` / `n_samples_cap` bound the per-step grid so it fits the T4; lower them first if you hit out-of-memory.

To train for RCT-style deployment, add `use_treatment=True, p_treatment_study=0.4` (note the README caveat — the treatment extension is not KS-validated).

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > T4 GPU"
torch.cuda.reset_peak_memory_stats()

model, history = train(
    n_steps=2000,          # demo length; serious runs need ~1M+ steps
    d=256,                 # try 384-512 if memory allows
    n_layers=6,
    n_heads=8,
    base_lr=3e-4,
    n_taxa_cap=512,        # subsample taxa per draw to fit the T4
    n_samples_cap=160,     # subsample samples per draw
    max_gen_taxa=600,      # cap the prior-GENERATED size (cuts CPU sampling cost)
    max_gen_samples=200,
    log_every=50,
    save_every=500,
    device_str="cuda",
    save_dir="checkpoints",
    use_amp=True,          # mixed precision (default on CUDA); pass False for fp32
    # --- For differential-abundance / treatment work, prefer this recipe: ---
    # use_treatment=True, p_treatment_study=0.5,   # treatment-aware prior
    # y_weight=0.0, effect_weight=0.0,             # the y head can't learn (see NOTES); focus the count head
    # (then judge the model on validation.ipynb experiment 6, not held-out rho)
)
print("final step loss:", history[-1]["loss"])
print(f"peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

## 4. Evaluate a checkpoint

Held-out cell prediction vs. the marginal-mean baseline (Spearman ρ, 80% coverage, NLL) on fresh prior draws.

In [ ]:
import glob
import os
import numpy as np
from microbiomepfn.eval import quick_eval

ckpt = max(glob.glob("checkpoints/model_step*.pt"), key=os.path.getmtime)
print("evaluating", ckpt)
res = quick_eval(ckpt, n_eval=5, device="cuda")
print("mean held-out NLL :", np.mean([r["nll_per_cell"] for r in res]))
print("model    rho      :", np.mean([r["spearman_model"] for r in res]))
print("baseline rho      :", np.mean([r["spearman_baseline"] for r in res]))

## 5. Scaling up & keeping your checkpoints

- **Longer / bigger runs:** push `n_steps` toward 1M+, and `d`/`n_layers` to `512`/`8`. On out-of-memory, lower `n_taxa_cap` / `n_samples_cap` first, then `d`.
- **Checkpoints** are written to `checkpoints/` (git-ignored) and Colab storage is **ephemeral** — mount Drive to keep them across sessions:
  ```python
  from google.colab import drive
  drive.mount('/content/drive')
  # then pass save_dir='/content/drive/MyDrive/microbiomepfn_ckpts' to train(...)
  ```
- **Resume / fine-tune** later with `microbiomepfn.finetune.finetune(pretrain_checkpoint=ckpt, trials=[...])`.
- **CLI equivalent:** `microbiomepfn-train --n_steps 100000 --d 512 --n_layers 8 --device cuda` (add `--no_amp` to force fp32).